In [1]:
using Pkg
Pkg.activate(".")
using DataDrivenConstrained

using DifferentialEquations
using ModelingToolkit
using DataDrivenDiffEq
using DataDrivenSparse
using MLJ
using Catalyst
using Plots
using DataFrames
using DataInterpolations
include("DataDrivenConstrained/examples/plotting.jl") # for plotting functions
export plot_interpolation, plot_data
using Statistics

  Activating project at `c:\Users\MGAJ\OneDrive - Danmarks Tekniske Universitet\DTU\Kandidat\5_Semester\Speciale\discovering_hidden_physics`
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Warning: Backwards compatability support of the new return codes to Symbols will be deprecated with the Julia v1.9 release. Please see https://docs.sciml.ai/SciMLBase/stable/interfaces/Solutions/#retcodes for more information
└ @ SciMLBase C:\Users\MGAJ\.julia\packages\SciMLBase\l4PVV\src\retcodes.jl:354
┌ Warning: Backwards compat

LoadError: LoadError: Failed to precompile DifferentialEquations [0c46a032-eb83-5123-abaf-570d42b7fbaa] to "C:\\Users\\MGAJ\\.julia\\compiled\\v1.10\\DifferentialEquations\\jl_8ED1.tmp".
in expression starting at c:\Users\MGAJ\OneDrive - Danmarks Tekniske Universitet\DTU\Kandidat\5_Semester\Speciale\discovering_hidden_physics\DataDrivenConstrained\src\DataDrivenConstrained.jl:1

In [7]:
using Lux
rbf(x) = exp.(-(x.^2))

# Multilayer FeedForward
U = Lux.Chain(
    Lux.Dense(2,5,rbf), Lux.Dense(5,5, rbf), Lux.Dense(5,5, rbf), Lux.Dense(5,2)
)

Chain(
    layer_1 = Dense(2 => 5, rbf),       # 15 parameters
    layer_2 = Dense(5 => 5, rbf),       # 30 parameters
    layer_3 = Dense(5 => 5, rbf),       # 30 parameters
    layer_4 = Dense(5 => 2),            # 12 parameters
)         # Total: 87 parameters,
          #        plus 0 states.

In [3]:
# ## Environment and packages
# #cd(@__DIR__)
# #using Pkg; Pkg.activate("."); Pkg.instantiate()

# #using OrdinaryDiffEq
# using ModelingToolkit
# using DataDrivenDiffEq
# using LinearAlgebra, ComponentArrays
# using Optimization   #OptimizationFlux for ADAM and OptimizationOptimJL for BFGS OptimizationOptimisers OptimizationOptimJL
# using DiffEqSensitivity
# using Lux
# using Plots
# gr()
# using JLD2, FileIO
# using Statistics
# # Set a random seed for reproduceable behaviour
# using Random
# rng = Random.default_rng()
# Random.seed!(1234)

In [ ]:
# ## Environment and packages
cd(@__DIR__)
using Pkg; Pkg.activate(".");
using Lux, Random, Optimisers, Zygote
# using LuxCUDA, AMDGPU, Metal, oneAPI # Optional packages for GPU support

# Seeding
rng = Random.default_rng()
Random.seed!(rng, 0)

# Construct the layer
model = Chain(Dense(128, 256, tanh), Chain(Dense(256, 1, tanh), Dense(1, 10)))

# Get the device determined by Lux
dev = gpu_device()

# Parameter and State Variables
ps, st = Lux.setup(rng, model) |> dev

# Dummy Input
x = rand(rng, Float32, 128, 2) |> dev

# Run the model
y, st = Lux.apply(model, x, ps, st)

# Gradients
## First construct a TrainState
train_state = Lux.Training.TrainState(model, ps, st, Adam(0.0001f0))

## We can compute the gradients using Training.compute_gradients
gs, loss, stats, train_state = Lux.Training.compute_gradients(AutoZygote(), MSELoss(),
    (x, dev(rand(rng, Float32, 10, 2))), train_state)

## Optimization
train_state = Lux.Training.apply_gradients(train_state, gs) # or Training.apply_gradients (no `!` at the end)

# Both these steps can be combined into a single call
gs, loss, stats, train_state = Training.single_train_step!(AutoZygote(), MSELoss(),
    (x, dev(rand(rng, Float32, 10, 2))), train_state)

  Activating project at `c:\Users\MGAJ\OneDrive - Danmarks Tekniske Universitet\DTU\Kandidat\5_Semester\Speciale\discovering_hidden_physics`
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Warning: Module Lux with build ID ffffffff-ffff-ffff-0000-4af908eae43c is missing from the cache.
│ This may mean Lux [b2108857-7c20-44ae-9111-449ecde12c47] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948


MethodError: MethodError: no method matching Lux.Experimental.TrainState(::Chain{@NamedTuple{layer_1::Dense{true, typeof(tanh_fast), typeof(glorot_uniform), typeof(zeros32)}, layer_2::Dense{true, typeof(tanh_fast), typeof(glorot_uniform), typeof(zeros32)}, layer_3::Dense{true, typeof(identity), typeof(glorot_uniform), typeof(zeros32)}}, Nothing}, ::@NamedTuple{layer_1::@NamedTuple{weight::Matrix{Float32}, bias::Matrix{Float32}}, layer_2::@NamedTuple{weight::Matrix{Float32}, bias::Matrix{Float32}}, layer_3::@NamedTuple{weight::Matrix{Float32}, bias::Matrix{Float32}}}, ::@NamedTuple{layer_1::@NamedTuple{}, layer_2::@NamedTuple{}, layer_3::@NamedTuple{}}, ::Adam)

Closest candidates are:
  Lux.Experimental.TrainState(::__T_model, ::__T_parameters, ::__T_states, ::__T_optimizer_state, !Matched::Int64) where {__T_model, __T_parameters, __T_states, __T_optimizer_state}
   @ Lux C:\Users\MGAJ\.julia\packages\ConcreteStructs\7Lv7u\src\ConcreteStructs.jl:142
